In [1]:
import copy
import os
import random
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model
from kryptonet import load_kryptonet

TARGET = 0.90


def load(name):
    """Load (X, y) arrays for a dataset."""
    return np.load(f"Datasets/{name}-X.npy"), np.load(f"Datasets/{name}-y.npy")


In [2]:
SEED = 42
RANKS = [1, 2, 4, 8, 16, 32, 64, 128, 256]
WEIGHT_DIR = "./weights/lora"


def set_seed(seed=42):
    """Set a seed for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_tensors(X, y):
    """Convert numpy arrays to float tensors."""
    return torch.from_numpy(X).float(), torch.from_numpy(y).float()


def accuracy(net, X, y):
    """Get accuracy of `net` on `X` and `y`."""
    net.eval()
    with torch.no_grad():
        return float(((net(X).squeeze(1) > 0).float() == y).float().mean())


# Adapted from MockPaperExperiments.ipynb
def fit(net, Xa, ya, Xv, yv, params, lr, epochs=300, patience=40, bs=128):
    """Adam on `params` with early stopping on a 20% validation split; restores the best weights."""
    opt = torch.optim.Adam(params, lr=lr)
    best, state, since = -1.0, None, 0
    for _ in range(epochs):
        net.train(); p = torch.randperm(len(ya))
        for i in range(0, len(ya), bs):
            bi = p[i:i + bs]
            loss = torch.nn.functional.binary_cross_entropy_with_logits(net(Xa[bi]).squeeze(1), ya[bi])
            opt.zero_grad(); loss.backward(); opt.step()
        va = accuracy(net, Xv, yv)
        if va > best:
            best, since, state = va, 0, copy.deepcopy(net.state_dict())
        else:
            since += 1
            if since >= patience:
                break
    net.load_state_dict(state)
    return net


def make_lora(rank):
    """Wrap the pretrained KryptoNet-40 with LoRA adapters of the given rank."""
    config = LoraConfig(
        r=rank,
        lora_alpha=2 * rank,
        target_modules=[str(n) for n in range(0, 9, 2)],  # linear layers: 0, 2, 4, 6, 8
        lora_dropout=0.05,
        bias="none",
    )
    return get_peft_model(load_kryptonet("kryptonet40.pt"), config)  # freezes the base weights


def split(X, y):
    """60% train, 20% validation, 20% test."""
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=SEED)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)
    return X_train, y_train, X_val, y_val, X_test, y_test


def rank_sweep(frac, ranks=RANKS, lr=1e-3):
    """Pick the best rank of LoRA on val of the first `frac`."""
    na, nv = int(round(frac * len(y_train))), int(round(frac * len(y_val)))
    Xa, ya = to_tensors(X_train[:na], y_train[:na])
    Xv, yv = to_tensors(X_val[:nv], y_val[:nv])
    out_dir = os.path.join(WEIGHT_DIR, f"{round(frac * 100)}pct")
    os.makedirs(out_dir, exist_ok=True)
    print(f"=== {frac:.0%} of train/val: {len(ya)} train / {len(yv)} val / {len(yt)} test ===")

    rows, best = [], None
    for r in ranks:
        set_seed(SEED)
        model = make_lora(r)
        trainable = [p for p in model.parameters() if p.requires_grad]
        model = fit(model, Xa, ya, Xv, yv, trainable, lr=lr)
        row = {"frac": frac, "rank": r, "params": sum(p.numel() for p in trainable),
               "train": accuracy(model, Xa, ya), "val": accuracy(model, Xv, yv)}
        rows.append(row)
        print(f"rank {r:>3} | params {row['params']:>9,} | train {row['train']:.4f} | val {row['val']:.4f}")
        if best is None or row["val"] > best["val"]:
            best, best_model = row, copy.deepcopy(model).merge_and_unload()

    # Save the best model for this fraction
    row = best
    row["test"] = accuracy(best_model, Xt, yt)
    row["state"] = best_model.state_dict()
    torch.save(row["state"], os.path.join(out_dir, "best.pt"))
    print(f"Best for {frac:.0%}: rank {row['rank']} (val {row['val']:.4f}) -> test {row['test']:.4f}, "
          f"saved {os.path.join(out_dir, 'best.pt')}")
    return rows


set_seed(SEED)
X, y = load("kryptonite-40")
X_train, y_train, X_val, y_val, X_test, y_test = split(X, y)
Xt, yt = to_tensors(X_test, y_test)
results = {}


## 100% of data

In [3]:
results[1.0] = rank_sweep(1.0)

=== 100% of train/val: 19200 train / 6400 val / 6400 test ===
rank   1 | params     4,137 | train 0.6054 | val 0.5173
rank   2 | params     8,274 | train 0.6967 | val 0.5272
rank   4 | params    16,548 | train 0.7765 | val 0.5437
rank   8 | params    33,096 | train 0.9689 | val 0.9484
rank  16 | params    66,192 | train 0.9636 | val 0.9544
rank  32 | params   132,384 | train 0.9642 | val 0.9539
rank  64 | params   264,768 | train 0.9618 | val 0.9511
rank 128 | params   529,536 | train 0.9621 | val 0.9552
rank 256 | params 1,059,072 | train 0.6214 | val 0.6170
Best for 100%: rank 128 (val 0.9552) -> test 0.9613, saved ./weights/lora\100pct\best.pt


## 50% of data

In [4]:
results[0.5] = rank_sweep(0.5)

=== 50% of train/val: 9600 train / 3200 val / 6400 test ===
rank   1 | params     4,137 | train 0.6009 | val 0.5125
rank   2 | params     8,274 | train 0.7418 | val 0.5191
rank   4 | params    16,548 | train 0.7840 | val 0.5188
rank   8 | params    33,096 | train 0.7578 | val 0.5263
rank  16 | params    66,192 | train 0.9882 | val 0.5359
rank  32 | params   132,384 | train 0.9804 | val 0.5378
rank  64 | params   264,768 | train 0.9990 | val 0.6856
rank 128 | params   529,536 | train 0.9780 | val 0.9278
rank 256 | params 1,059,072 | train 0.9604 | val 0.9459
Best for 50%: rank 256 (val 0.9459) -> test 0.9508, saved ./weights/lora\50pct\best.pt


## 25% of data

In [5]:
results[0.25] = rank_sweep(0.25)

=== 25% of train/val: 4800 train / 1600 val / 6400 test ===
rank   1 | params     4,137 | train 0.5502 | val 0.5050
rank   2 | params     8,274 | train 0.5148 | val 0.5088
rank   4 | params    16,548 | train 0.5967 | val 0.5075
rank   8 | params    33,096 | train 0.9540 | val 0.5231
rank  16 | params    66,192 | train 0.5285 | val 0.5094
rank  32 | params   132,384 | train 0.9944 | val 0.5306
rank  64 | params   264,768 | train 0.9706 | val 0.5406
rank 128 | params   529,536 | train 0.9892 | val 0.5288
rank 256 | params 1,059,072 | train 0.9290 | val 0.5469
Best for 25%: rank 256 (val 0.5469) -> test 0.5241, saved ./weights/lora\25pct\best.pt


### Summary

In [6]:
print(f"{'subset':>6} | {'best rank':>9} | {'train':>6} | {'val':>6} | {'test':>6}")
for frac, rows in results.items():
    b = next(r for r in rows if "state" in r)
    print(f"{frac:>6.0%} | {b['rank']:>9} | {b['train']:.4f} | {b['val']:.4f} | {b['test']:.4f}")

# The best across subsets chosen by validation accuracy among each subset's ranks
overall = max((r for rows in results.values() for r in rows if "state" in r), key=lambda r: r["val"])
torch.save(overall["state"], os.path.join(WEIGHT_DIR, "best.pt"))
print()
print(f"Overall best: {overall['frac']:.0%} subset, rank {overall['rank']} "
      f"(val {overall['val']:.4f}, test {overall['test']:.4f}) -> saved {os.path.join(WEIGHT_DIR, 'best.pt')}")


subset | best rank |  train |    val |   test
  100% |       128 | 0.9621 | 0.9552 | 0.9613
   50% |       256 | 0.9604 | 0.9459 | 0.9508
   25% |       256 | 0.9290 | 0.5469 | 0.5241

Overall best: 100% subset, rank 128 (val 0.9552, test 0.9613) -> saved ./weights/lora\best.pt


## LoRA on Kryptonite-40-Adapt

In [7]:
def rank_sweep_lite(dataset, keyword, ranks=RANKS, lr=1e-3):
    """Lite version of rank sweep"""
    Xd, yd = load(dataset)
    Xa, ya, Xv, yv, Xte, yte = split(Xd, yd)
    (Xa, ya), (Xv, yv), (Xte, yte) = to_tensors(Xa, ya), to_tensors(Xv, yv), to_tensors(Xte, yte)
    out_dir = os.path.join(WEIGHT_DIR, keyword)
    os.makedirs(out_dir, exist_ok=True)
    print(f"=== {dataset}: {len(ya)} train / {len(yv)} val / {len(yte)} test ===")

    rows, best = [], None
    for r in ranks:
        set_seed(SEED)
        model = make_lora(r)
        trainable = [p for p in model.parameters() if p.requires_grad]
        model = fit(model, Xa, ya, Xv, yv, trainable, lr=lr)
        row = {"dataset": dataset, "rank": r, "params": sum(p.numel() for p in trainable),
               "train": accuracy(model, Xa, ya), "val": accuracy(model, Xv, yv)}
        rows.append(row)
        print(f"rank {r:>3} | params {row['params']:>9,} | train {row['train']:.4f} | val {row['val']:.4f}")
        if best is None or row["val"] > best["val"]:
            best, best_model = row, copy.deepcopy(model).merge_and_unload()

    best["test"] = accuracy(best_model, Xte, yte)
    best["state"] = best_model.state_dict()
    path = os.path.join(out_dir, "best.pt")
    torch.save(best["state"], path)
    print(f"Best for {dataset}: rank {best['rank']} (val {best['val']:.4f}) -> test {best['test']:.4f}, saved {path}")
    return rows

adapt_results = rank_sweep_lite("kryptonite-40-adapt", "adapt", ranks=RANKS)


=== kryptonite-40-adapt: 2400 train / 800 val / 800 test ===
rank   1 | params     4,137 | train 0.9742 | val 0.9625
rank   2 | params     8,274 | train 0.9754 | val 0.9388
rank   4 | params    16,548 | train 0.9742 | val 0.9588
rank   8 | params    33,096 | train 0.9725 | val 0.9538
rank  16 | params    66,192 | train 0.9712 | val 0.9613
rank  32 | params   132,384 | train 1.0000 | val 0.9550
rank  64 | params   264,768 | train 1.0000 | val 0.9400
rank 128 | params   529,536 | train 0.9996 | val 0.9500
rank 256 | params 1,059,072 | train 0.9608 | val 0.5337
Best for kryptonite-40-adapt: rank 1 (val 0.9625) -> test 0.9475, saved ./weights/lora\adapt\best.pt


## Predict hidden labels

In [8]:
HIDDEN_DIR = "./hiddenlabels"


def predict_hidden(weights, dataset, filename):
    """Predict labels for a hidden dataset and save them to HIDDEN_DIR."""
    model = load_kryptonet(weights)
    Xh = np.load(f"Datasets/{dataset}-X.npy")
    with torch.no_grad():
        labels = (model(torch.from_numpy(Xh).float()).squeeze(1) > 0).long().numpy()
    os.makedirs(HIDDEN_DIR, exist_ok=True)
    path = os.path.join(HIDDEN_DIR, filename)
    np.save(path, labels)
    print(f"{weights} -> {path} | shape {labels.shape} | fraction of 1s {labels.mean():.4f}")
    return labels

hidden_40 = predict_hidden(os.path.join(WEIGHT_DIR, "100pct", "best.pt"), "hidden-kryptonite-40", "y_predicted_40.npy")
hidden_40_adapt = predict_hidden(os.path.join(WEIGHT_DIR, "adapt", "best.pt"), "hidden-kryptonite-40-adapt",
                                 "y_predicted_40_adapt.npy")


./weights/lora\100pct\best.pt -> ./hiddenlabels\y_predicted_40.npy | shape (10000,) | fraction of 1s 0.5039
./weights/lora\adapt\best.pt -> ./hiddenlabels\y_predicted_40_adapt.npy | shape (10000,) | fraction of 1s 0.5142
